In [ ]:
%pip install -q lightgbm xgboost
import shutil
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, roc_auc_score)

In [ ]:
# входы пересобираются из датасета каждым циклом обучения (train_inputs)
TI = Path("data/dataset/train_inputs")
USE_GPU = shutil.which("nvidia-smi") is not None
SEED, VAL_FRAC = 0, 0.2
FEATURES = ["sulf_fill", "gray_fill", "dark_fill", "sulf_solidity",
            "frag_density", "thick_rel", "extent", "log_area_rel",
            "core_fill", "core_frag"]

def make_model(n_estimators, lr):
    # GPU -> XGBoost CUDA, CPU -> LightGBM
    if USE_GPU:
        from xgboost import XGBClassifier
        return XGBClassifier(n_estimators=n_estimators, learning_rate=lr,
                             max_depth=7, subsample=0.8, colsample_bytree=0.8,
                             tree_method="hist", device="cuda",
                             random_state=SEED, eval_metric="logloss")
    import lightgbm as lgb
    return lgb.LGBMClassifier(n_estimators=n_estimators, learning_rate=lr,
                              num_leaves=31, subsample=0.8, colsample_bytree=0.8,
                              random_state=SEED, verbose=-1)

def to_cpu(m):
    # веса должны работать на CPU при инференсе
    if hasattr(m, "set_params") and "device" in m.get_params():
        m.set_params(device="cpu")
    return m

print("GPU:", USE_GPU)

In [ ]:
# срастания: обычное (0) vs тонкое (1) зерно, слабые метки, групповой сплит
df = pd.read_csv(TI / "grains.csv")
rng = np.random.RandomState(SEED)
groups = sorted(df.group.unique())
val_groups = set(rng.choice(groups, int(len(groups) * VAL_FRAC), replace=False))
tr, va = df[~df.group.isin(val_groups)], df[df.group.isin(val_groups)]
print(f"train {len(tr)} / val {len(va)} зёрен")

model_g = make_model(600, 0.05)
model_g.fit(tr[FEATURES].values, tr.weak_label.values,
            sample_weight=np.sqrt(tr.area.values))
proba = to_cpu(model_g).predict_proba(va[FEATURES].values)[:, 1]
print(f"зёрна: acc={((proba>=0.5)==va.weak_label).mean():.3f} "
      f"AUC={roc_auc_score(va.weak_label, proba):.3f}")

img = []
for (path, cls), sub in va.groupby(["path", "cls"]):
    p = model_g.predict_proba(sub[FEATURES].values)[:, 1]
    a = sub.area.values.astype(float)
    img.append((cls == "труднообогатимая", a[p >= 0.5].sum() / a.sum()))
img = pd.DataFrame(img, columns=["y", "fine_share"])
y_pred = img.fine_share > 0.5
print(f"изображения: macro-F1={f1_score(img.y, y_pred, average='macro'):.3f} "
      f"AUC={roc_auc_score(img.y, img.fine_share):.3f}")
print(confusion_matrix(img.y, y_pred))

joblib.dump(model_g, "model_grains.joblib")
# результат: model_grains.joblib -> artifacts/intergrowth/model.joblib

In [ ]:
# тальк: пиксельный классификатор
d = np.load(TI / "pixels.npz", allow_pickle=True)
X, y, grp = d["X"], d["y"], d["groups"]
ug = np.unique(grp)
val_g = set(np.random.RandomState(SEED).choice(ug, max(int(len(ug)*0.2), 2), replace=False))
val_m = np.isin(grp, list(val_g))

model_t = make_model(200, 0.08)
model_t.fit(X[~val_m], y[~val_m])
p = to_cpu(model_t).predict_proba(X[val_m])[:, 1]
print(f"пиксели (val): AUC={roc_auc_score(y[val_m], p):.3f} "
      f"acc={((p>=0.5)==y[val_m]).mean():.3f}")

joblib.dump(model_t, "model_talc.joblib")
# результат: model_talc.joblib -> artifacts/talc/model.joblib